# 04 - Feature Engineering

This notebook creates analysis-ready features from the cleaned DVF sample.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dvf_clean_sample.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Original shape:", df.shape)
df.head()

## Date Features

Extract year and month from the transaction date.

In [ ]:
df["Date mutation"] = pd.to_datetime(df["Date mutation"], errors="coerce")
df["year"] = df["Date mutation"].dt.year
df["month"] = df["Date mutation"].dt.month

df[["Date mutation", "year", "month"]].head()

## Price Per Square Meter

Create `price_per_m2` from `Valeur fonciere` and `Surface reelle bati` where the built surface is available and positive.

In [ ]:
surface_col = "Surface reelle bati"

if surface_col in df.columns:
    df[surface_col] = pd.to_numeric(df[surface_col], errors="coerce")
    df["price_per_m2"] = np.where(
        df[surface_col] > 0,
        df["Valeur fonciere"] / df[surface_col],
        np.nan
    )
else:
    df[surface_col] = np.nan
    df["price_per_m2"] = np.nan

df[["Valeur fonciere", surface_col, "price_per_m2"]].head()

## Department Feature

The French department is extracted from the postal code.

In [ ]:
postal_col = "Code postal"

if postal_col in df.columns:
    postal_code = df[postal_col].astype("Int64").astype(str).str.zfill(5)
    df["department"] = postal_code.str[:2]
else:
    df[postal_col] = pd.NA
    df["department"] = pd.NA

df[[postal_col, "department"]].head()

## Save Feature Dataset

In [ ]:
output_path = PROJECT_ROOT / "data" / "processed" / "dvf_features_sample.csv"
df.to_csv(output_path, index=False)

print("Saved feature dataset to:")
print(output_path)
print("Final shape:", df.shape)